# Province base data extraction
Province id, name, development, trade good, center of trade level, culture and religion

In [4]:
import os
import pandas as pd
import re

In [5]:
DESIRED_PROV_DATA = ['owner', 'culture', 'religion', 'hre', 'base_tax', 'base_production', 'trade_goods', 'base_manpower', 'capital', 'is_city', 'center_of_trade']

In [ ]:
EU4_DIR = os.getenv("EU4_INSTALL_LOCATION")
SEPARATOR = os.path.sep
PROV_HISTORY_DIR = EU4_DIR + SEPARATOR + 'history' + SEPARATOR + 'provinces'
assert(os.path.isdir(EU4_DIR))

print(os.listdir(PROV_HISTORY_DIR))

['1-Uppland.txt', '10-Jamtland.txt', '100 - Friesland.txt', '1000 - Cree.txt', '1001 - Sault.txt', '1002 - Abitbi.txt', '1003 - Moose Cree.txt', '1004 - Attawapiskat.txt', '1005 - Swampy Cree.txt', '1006 - Chipewyan.txt', '1007 - Ojibwa.txt', '1008 - Winnipeg.txt', '1009 - Atikaki.txt', '101 - Genoa.txt', '1010 - Manitoba.txt', '1011 - Woods Cree.txt', '1012 - Satsuma.txt', '1013 - Cheongju.txt', '1014 - Bungo.txt', '1015 - Okinawa.txt', '1016 - Ha Tinh.txt', '1017 - Suo.txt', '1018 - Izumo.txt', '1019 - Harima.txt', '102 - Nice.txt', '1020 - Yamashiro.txt', '1021 - Settsu.txt', '1022 - Indrapura.txt', '1023 - Echizen.txt', '1024 - Echigo.txt', '1025 - Dewa.txt', '1026 - Rikuzen.txt', '1027 - Hitachi.txt', '1028 - Musashi.txt', '1029 - Kai.txt', '103 - Piedmont.txt', '1030 - Owari.txt', '1031 - Kamikawa.txt', '1032 - Kurils.txt', '1033 - Sakhalin.txt', '1034 - Kamchatka.txt', '1035 - Penchisky.txt', '1036 - Podzhiversk.txt', '1037 - Nizhe-Kolymsk.txt', '1038 - Anadyrsk.txt', '1039 - Ve

In [5]:
def clean_and_extract_filename(filename):
    if '-' not in filename:
        reg = re.compile(r'\D\w+')
        name_start = re.search(reg, filename).start()
        filename = filename[:name_start] + '-' + filename[name_start:]
    path_no_spaces = filename.replace(' ', '')
    path_no_ext = path_no_spaces.split('.')[0]
    path_id_name = path_no_ext.split('-')
    province_id = path_id_name[0]
    province_name = path_id_name[1]
    return filename, province_id, province_name

In [6]:
def identify_and_clean_line(line):
    line = line.strip()
    line = line.replace('"', '')
    if '#' in line:
        return None, None
    if len(line) < 1:
        return None, None
    args = line.split(' ')
    if len(args) != 3:
        return None, None
    key = args[0]
    value = args[2]
    return key, value

In [7]:
def read_province_file(path, filename):
    filename, province_id, province_name = clean_and_extract_filename(filename)
    province_dict = {}
    with open(path) as file:
        for line in file:
            key, value = identify_and_clean_line(line)
            if key in DESIRED_PROV_DATA and key not in province_dict.keys():
                province_dict.update({key: value})
    for expected_key in DESIRED_PROV_DATA:
        if expected_key not in province_dict.keys():
            province_dict.update({expected_key: None})
    province_dict.update({'name': province_name, 'id': province_id})
    return province_dict

In [8]:
data = []
for file in os.listdir(PROV_HISTORY_DIR):
    province_dict = read_province_file(PROV_HISTORY_DIR + SEPARATOR + file, file)
    data.append(province_dict)
df = pd.DataFrame(data)

In [9]:
df.index = df.index.astype(int)
df = df.sort_index()
df.head()

,owner,culture,religion,hre,base_tax,base_production,trade_goods,base_manpower,capital,is_city,center_of_trade,name,id
0,SWE,swedish,catholic,no,5,5,grain,3,Stockholm,yes,2,Uppland,1
1,NOR,NaN,catholic,no,1,1,fur,1,Frösön,yes,NaN,Jamtland,10
2,FRI,frisian,catholic,yes,6,6,livestock,2,Leeuwarden,yes,NaN,Friesland,100
3,ENG,cree,totemism,no,1,2,unknown,1,Cree,yes,NaN,Cree,1000
4,FRA,anishinabe,totemism,no,2,2,unknown,1,Sault,yes,NaN,Sault,1001


In [10]:
df.isna().sum()

owner               887
culture             686
religion            868
hre                 651
base_tax            652
base_production     656
trade_goods         761
base_manpower       653
capital             893
is_city             989
center_of_trade    3561
name                  0
id                    0
dtype: int64

In [11]:
df['owner'] = df['owner'].fillna('Unowned')
df['culture'] = df['culture'].fillna('Unowned')
df['religion'] = df['religion'].fillna('Unowned')
df['hre'] = df['hre'].fillna('no')
df['base_tax'] = df['base_tax'].fillna(0)
df['base_production'] = df['base_production'].fillna(0)
df['base_manpower'] = df['base_manpower'].fillna(0)
df['capital'] = df['capital'].fillna(df['name'])
df['is_city'] = df['is_city'].fillna('no')
df['center_of_trade'] = df['center_of_trade'].fillna(0)

## Trade nodes

In [12]:
def process_tradenode_file_line(line, node_data, inland, end_node, location, members, trade_node, in_members_segment):
    line = line.strip()
    if 'in_members_segment' not in locals():
        in_members_segment = False
    if 'trade_node' not in locals():
        trade_node = ''
    if re.match(node_name_regex, line):
        if len(trade_node) > 0:
            node_data.append([trade_node, inland, end_node, len(members), location])
        # Initialise all node values
        inland = False
        end_node = False
        location = None
        members = []
        trade_node = line.strip().rstrip('={') # Remove whitespace before removing ={
        return node_data, inland, end_node, location, members, trade_node, in_members_segment
    if re.match(members_list_regex, line):
        in_members_segment = True
        return node_data, inland, end_node, location, members, trade_node, in_members_segment
    if in_members_segment and re.match(numbers_in_members_list_regex, line):
        for member in line.strip().split(' '):
            members.append(member)
        for member in members:
            df.loc[df['id'] == member, 'trade_node'] = trade_node
        in_members_segment = False
        return node_data, inland, end_node, location, members, trade_node, in_members_segment
    try:
        split_line = line.split('=')
    except:
        assert('=' not in line)
    if split_line:
        if split_line[0].strip() == 'inland' and split_line[1].strip() == 'yes':
            inland = True
            return node_data, inland, end_node, location, members, trade_node, in_members_segment
        if split_line[0].strip() == 'end' and split_line[1].strip() == 'yes':
            end_node = True
            return node_data, inland, end_node, location, members, trade_node, in_members_segment
        if split_line[0].strip() == 'location':
            location = int(split_line[1].strip())
    if in_members_segment and '}' in line and not re.match(numbers_in_members_list_regex, line):
        in_members_segment = False
    return node_data, inland, end_node, location, members, trade_node, in_members_segment

In [13]:
def initialise_tradenode_context():
    inland = False
    end_node = False
    location = None
    in_members_segment = False
    members = []
    node_data = []
    trade_node = ''
    return inland, end_node, location, in_members_segment, members, node_data, trade_node

In [14]:
trade_nodes_path = EU4_DIR + SEPARATOR + 'common' + SEPARATOR + 'tradenodes' + SEPARATOR + '00_tradenodes.txt'
node_name_regex = re.compile(r'\b(?!color\b)(?!outgoing\b)(?!path\b)(?!control\b)(?!members\b)\w+=\{')
members_list_regex = re.compile(r'(members={)')
numbers_in_members_list_regex = re.compile(r'\w+(\d+)\w')
df['trade_node'] = None
inland, end_node, location, in_members_segment, members, node_data, trade_node = initialise_tradenode_context()
with open(trade_nodes_path) as nodefile:
    for line in nodefile:
        node_data, inland, end_node, location, members, trade_node, in_members_segment = process_tradenode_file_line(line, node_data, inland, end_node, location, members, trade_node, in_members_segment)
df_trade_nodes = pd.DataFrame(data=node_data, columns=['name', 'inland', 'end_node', 'member_count', 'location'])

In [15]:
df_trade_nodes

,name,inland,end_node,member_count,location
0,african_great_lakes,True,False,30,4064
1,kongo,True,False,32,4097
2,zambezi,True,False,29,1191
3,patagonia,False,False,20,2868
4,amazonas_node,False,False,37,2935
...,...,...,...,...,...
74,sevilla,False,False,53,1293
75,champagne,True,False,33,186
76,valencia,False,False,22,1295
77,genua,False,True,42,1298


In [16]:
df.loc[df["owner"] == 'MAL']

,owner,culture,religion,hre,base_tax,base_production,trade_goods,base_manpower,capital,is_city,center_of_trade,name,id,trade_node
129,MAL,senegambian,Unowned,no,3,3,ivory,2,Kansala,yes,1,Gabu,1114,ivory_coast
132,MAL,fulani,Unowned,no,3,3,ivory,3,Timbo,yes,0,FutaJallon,1117,ivory_coast
136,MAL,soninke,Unowned,no,6,6,gold,2,Bambuk,yes,0,Bambuk,1120,timbuktu
137,MAL,mali,Unowned,no,6,6,gold,2,Kurussa,yes,0,Bure,1121,timbuktu
138,MAL,dyola,Unowned,no,1,1,dyes,1,Tingrela,yes,0,Bagoe,1122,timbuktu
139,MAL,bambara,Unowned,no,5,5,grain,2,Kirina,yes,0,Segu,1123,timbuktu
140,MAL,mali,Unowned,no,4,4,grain,8,Niani,yes,1,Wasuju,1124,timbuktu
1376,MAL,senegambian,Unowned,no,1,1,ivory,1,Sutuco,yes,0,Kantor,2238,ivory_coast
1382,MAL,soninke,sunni,no,1,1,livestock,1,Awdaghust,yes,0,Termes,2243,timbuktu
1384,MAL,soninke,Unowned,no,2,2,wool,1,Dia,yes,0,Wagadu,2245,timbuktu


In [17]:
df.to_csv('./data/provinces_stage1.csv')